# 本地纯文本 + LangChain 工具（mock）

与 `20_local_vision_tools_mock.ipynb` 同一套 **LM Studio / OpenAI 兼容**端点，本文件**仅文本**，不涉及图片或 Vision。

- 服务根地址一般为 `http://127.0.0.1:1234`，Python 里 **base_url** 请设为 `http://127.0.0.1:1234/v1`（OpenAI 兼容路由）。
- 请加载**支持 function / tool calling** 的纯文本模型；勿依赖 VL 仅视觉模型（除非你确认其同样支持工具调用）。
- **「思考」**：若服务端未返回独立推理字段，则主要体现在模型的自然语言 `content` 与是否发起 `tool_calls`；部分模型会在 `additional_kwargs` / `response_metadata` 中带扩展字段，下面代码会尽量打印。

In [ ]:
%pip install -q "langchain-core>=0.3" "langchain-openai>=0.2"

In [ ]:
import json

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

BASE_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
# 改为当前服务上已加载的、支持 tool calling 的纯文本模型 id
MODEL = "replace-with-your-text-model-id"


@tool
def get_mock_weather(region: str) -> str:
    """查询某地区的天气概况。当前为占位实现，不真实请求气象服务。"""
    return f"[MOCK 天气] {region}: 晴间多云，10–15°C，南风 2 级"


@tool
def lookup_symbol_hint(text: str) -> str:
    """根据用户提到的地名或关键词返回简短场景标签。占位实现。"""
    return f"[MOCK 标签] {text!r} -> 高原, 旅行, 户外"


tools = [get_mock_weather, lookup_symbol_hint]
tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)


def _fmt_extra(m: BaseMessage) -> str:
    parts: list[str] = []
    ak = getattr(m, "additional_kwargs", None) or {}
    if ak:
        parts.append(f"additional_kwargs: {ak}")
    rm = getattr(m, "response_metadata", None) or {}
    if rm:
        parts.append(f"response_metadata: {rm}")
    return "\n    " + "\n    ".join(parts) if parts else ""


def print_message(tag: str, m: BaseMessage) -> None:
    print(f"\n--- [{tag}] {m.__class__.__name__} ---")
    if isinstance(m, HumanMessage):
        print(m.content)
    elif isinstance(m, AIMessage):
        if m.content:
            print("content:\n", m.content)
        tcs = getattr(m, "tool_calls", None) or []
        if tcs:
            print("tool_calls (判断):")
            for tc in tcs:
                print(f"  - {tc['name']!r} id={tc['id']!r} args={tc.get('args', tc.get('arguments'))}")
        extra = _fmt_extra(m)
        if extra:
            print(extra.strip())
    elif isinstance(m, ToolMessage):
        print(f"tool_call_id: {m.tool_call_id}")
        print(m.content)
    else:
        print(m)


user_text = (
    "用户打算去青藏高原旅行。请先调用 lookup_symbol_hint，text 用简短中文关键词；"
    "再调用 get_mock_weather，region 填「青藏高原」或你根据上下文确定的地区名。"
    "最后用中文汇总工具返回，不要编造工具未给出的数字。"
)
messages: list[BaseMessage] = [HumanMessage(content=user_text)]
print_message("用户", messages[0])

MAX_ROUNDS = 5
for round_i in range(MAX_ROUNDS):
    ai: AIMessage = llm_with_tools.invoke(messages)
    print_message(f"模型 第{round_i + 1}轮", ai)
    messages.append(ai)

    tool_calls = getattr(ai, "tool_calls", None) or []
    if not tool_calls:
        if not ai.content:
            print("(本轮无正文且无 tool_calls)")
        break

    for tc in tool_calls:
        name = tc["name"]
        raw = tc.get("args")
        if raw is None:
            raw = tc.get("arguments", {})
        if isinstance(raw, str):
            raw = json.loads(raw) if raw.strip() else {}
        if not isinstance(raw, dict):
            raw = {}
        out = tools_by_name[name].invoke(raw)
        tm = ToolMessage(content=str(out), tool_call_id=tc["id"])
        print_message(f"工具 → {name}", tm)
        messages.append(tm)
else:
    raise RuntimeError(f"超过 MAX_ROUNDS={MAX_ROUNDS}，请检查模型是否陷入重复 tool 调用。")

print("\n=== 结束 ===")